# Akkadian → English NMT — v3 Final
### Research Pipeline | Deep Past Challenge

**Key change from v2:** All available datasets are fully mined for parallel pairs.  
Previous run had only **1,654 training pairs** → BLEU 0.10.  
This version targets **5,000–15,000+ pairs** → BLEU **0.32–0.40**.

### Data sources used
| File | Role |
|------|------|
| Deep Past `train.csv` | Primary competition domain |
| ORACC `train.csv` | Large general Akkadian–English corpus |
| `Sentences_Oare_FirstWord_LinNum.csv` | Extra sentence pairs (mined) |
| `eBL_Dictionary.csv` + `OA_Lexicon_eBL.csv` | Glossary for OOV fallback |
| `published_texts.csv` | Mined for additional pairs if columns allow |

### Training improvements
- Label smoothing ε=0.1
- Cosine-with-restarts LR schedule
- Aggressive data augmentation (word-swap + back-noise)
- Larger beam (k=10) + repetition penalty at inference
- Early stopping patience=6

In [1]:
# ═══════════════════════════════════════════════════════════════════
# 0. INSTALL DEPENDENCIES
# Run this cell once, then: Run → Restart & Run All
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys

def pip(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

pip('numpy>=2.0')      # pin numpy FIRST — prevents binary conflicts
pip('transformers')
pip('sentencepiece')
pip('datasets')
pip('accelerate')
pip('sacrebleu')       # no [ja] extra — avoids ipadic dragging old numpy
pip('evaluate')
pip('rouge-score')
pip('bert-score')
pip('nltk')
pip('scikit-learn')

print('All packages installed.')
print('Now: Restart & Run All')

All packages installed.
Now: Restart & Run All


In [2]:
# ═══════════════════════════════════════════════════════════════════
# 1. IMPORTS & SETUP
# ═══════════════════════════════════════════════════════════════════
import nltk
nltk.download('punkt',   quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

import os, re, math, random, json, warnings, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

warnings.filterwarnings('ignore')
print('NumPy  :', np.__version__)   # must be 2.x
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

NumPy  : 2.4.4
Torch  : 2.10.0+cu128
CUDA   : True
Device : cuda


In [3]:
# ═══════════════════════════════════════════════════════════════════
# 2. PATHS & DISCOVERY
# ═══════════════════════════════════════════════════════════════════
DEEP_PAST_DIR = Path('/kaggle/input/competitions/deep-past-initiative-machine-translation')
ORACC_DIR     = Path('/kaggle/input/datasets/manwithacat/oracc-akkadian-english-parallel-corpus')

print('Deep Past files:')
for f in sorted(os.listdir(DEEP_PAST_DIR)):
    size = os.path.getsize(DEEP_PAST_DIR / f) // 1024
    print(f'  {f:45s} {size:>6} KB')

print('\nORACC files:')
for f in sorted(os.listdir(ORACC_DIR)):
    size = os.path.getsize(ORACC_DIR / f) // 1024
    print(f'  {f:45s} {size:>6} KB')

Deep Past files:
  OA_Lexicon_eBL.csv                              3470 KB
  Sentences_Oare_FirstWord_LinNum.csv             1939 KB
  bibliography.csv                                 148 KB
  eBL_Dictionary.csv                              1582 KB
  publications.csv                              566991 KB
  published_texts.csv                            10878 KB
  resources.csv                                     94 KB
  sample_submission.csv                              0 KB
  test.csv                                           0 KB
  train.csv                                       1589 KB

ORACC files:
  statistics.json                                    0 KB
  train.csv                                       8118 KB


In [4]:
# ═══════════════════════════════════════════════════════════════════
# 3. NORMALISATION HELPERS  (defined early — needed by all loaders)
# ═══════════════════════════════════════════════════════════════════
SUBSCRIPT_MAP   = str.maketrans('₀₁₂₃₄₅₆₇₈₉', '0123456789')
SUPERSCRIPT_MAP = str.maketrans('⁰¹²³⁴⁵⁶⁷⁸⁹', '0123456789')
DIACRITIC_MAP   = str.maketrans('ṣṭšŠṢṬṭ', 'stsSSТt')

def normalize_akkadian(text: str) -> str:
    text = str(text)
    text = text.translate(SUBSCRIPT_MAP).translate(SUPERSCRIPT_MAP)
    text = text.translate(DIACRITIC_MAP)
    text = text.replace('ḫ', 'h').replace('Ḫ', 'H')
    text = text.replace('…', ' <gap> ')
    text = re.sub(r'\.{2,}', ' <gap> ', text)
    text = re.sub(r'\[([^\]]*?)\]', r'\1', text)          # keep restoration content
    text = re.sub(r'<[^>]+>', ' ', text)                   # drop angle brackets
    text = re.sub(r'\{d\}',   ' DEITY ', text)
    text = re.sub(r'\{ki\}',  ' PLACE ', text)
    text = re.sub(r'\{f\}',   ' FEM ',   text)
    text = re.sub(r'\{[^}]+\}', ' DET ', text)
    text = re.sub(r'\bx\b', '<unk_sign>', text, flags=re.IGNORECASE)
    text = text.replace('-', ' ')
    text = re.sub(r'[!?/:;]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_english(text: str) -> str:
    text = str(text)
    text = text.replace('\u2014', ' - ').replace('\u2013', ' - ')
    text = re.sub(r'_gap_+',      ' <gap> ', text)
    text = re.sub(r'<gap\s*/?>', ' <gap> ', text)
    text = re.sub(r'\.{2,}',     ' <gap> ', text)
    text = re.sub(r'\[([^\]]*?)\]', r'\1', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Normalisation helpers defined.')

Normalisation helpers defined.


In [5]:
# ═══════════════════════════════════════════════════════════════════
# 4. QUALITY FILTER
# ═══════════════════════════════════════════════════════════════════
def has_repeated_phrases(tokens, n=4, threshold=2):
    if len(tokens) < n * threshold:
        return False
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return any(v >= threshold for v in Counter(ngrams).values())

def is_bad_pair(akk: str, eng: str) -> str:
    if not akk or not eng:
        return 'empty'
    akk_tok, eng_tok = akk.split(), eng.split()
    n_akk, n_eng = len(akk_tok), len(eng_tok)
    if n_akk < 2 or n_eng < 2:          return 'too_short'
    if n_akk > 250 or n_eng > 250:      return 'too_long'
    ratio = n_eng / max(n_akk, 1)
    if ratio < 0.12 or ratio > 7.0:     return f'length_ratio_{ratio:.2f}'
    gap_frac = eng_tok.count('<gap>') / max(n_eng, 1)
    if gap_frac > 0.45:                 return 'mostly_gaps'
    if re.fullmatch(r'[\d\s\.,;:!?()-]+', eng): return 'no_words'
    if has_repeated_phrases(eng_tok, n=4, threshold=2): return 'repeated_phrases'
    if eng.count('-') / max(len(eng), 1) > 0.15: return 'likely_transliteration'
    return ''

def clean_df(df: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df.copy()
    df['akkadian'] = df['akkadian'].fillna('').apply(normalize_akkadian)
    df['english']  = df['english'].fillna('').apply(normalize_english)
    reasons  = df.apply(lambda r: is_bad_pair(r['akkadian'], r['english']), axis=1)
    bad_mask = reasons != ''
    df = df[~bad_mask].reset_index(drop=True)
    before = len(df)
    df = df.drop_duplicates(subset=['akkadian', 'english']).reset_index(drop=True)
    print(f'{name}: {len(df)} clean pairs  (dropped {bad_mask.sum()} bad, {before-len(df)} dupes)')
    return df[['akkadian', 'english']]

print('Quality filter defined.')

Quality filter defined.


In [6]:
# ═══════════════════════════════════════════════════════════════════
# 5. LOAD ALL DATASETS
# ═══════════════════════════════════════════════════════════════════

all_frames = []

# ── 5A. Deep Past train.csv (primary competition domain) ────────────
train_raw = pd.read_csv(DEEP_PAST_DIR / 'train.csv')
train_raw = train_raw.rename(columns={'transliteration': 'akkadian', 'translation': 'english'})
train_raw = train_raw[['akkadian', 'english']]
comp_clean = clean_df(train_raw, 'Deep Past train')
all_frames.append(('competition', comp_clean))

# ── 5B. ORACC train.csv ─────────────────────────────────────────────
oracc_raw  = pd.read_csv(ORACC_DIR / 'train.csv')
oracc_clean = clean_df(oracc_raw[['akkadian', 'english']], 'ORACC')
all_frames.append(('oracc', oracc_clean))

# ── 5C. Sentences_Oare_FirstWord_LinNum.csv ─────────────────────────
# Inspect columns first
try:
    sent_df = pd.read_csv(DEEP_PAST_DIR / 'Sentences_Oare_FirstWord_LinNum.csv')
    print('\nSentences_Oare columns:', sent_df.columns.tolist())
    print('Shape:', sent_df.shape)
    print(sent_df.head(3).to_string())
    HAS_SENTENCES = True
except Exception as e:
    print('Sentences file error:', e)
    HAS_SENTENCES = False

Deep Past train: 987 clean pairs  (dropped 574 bad, 0 dupes)
ORACC: 831 clean pairs  (dropped 1284 bad, 2 dupes)

Sentences_Oare columns: ['display_name', 'text_uuid', 'sentence_uuid', 'sentence_obj_in_text', 'translation', 'first_word_transcription', 'first_word_spelling', 'first_word_number', 'first_word_obj_in_text', 'line_number', 'side', 'column']
Shape: (9782, 12)
             display_name                             text_uuid                         sentence_uuid  sentence_obj_in_text                                                             translation first_word_transcription first_word_spelling  first_word_number  first_word_obj_in_text  line_number  side  column
0   (Adana 237a) envelope  871e7671-e26b-4808-9acb-1c4a7e1ed2e6  f98879ad-d891-4602-8443-965baab6845d                     3                                     Seal: trading station of Šalatiwar.                      NaN               KIŠIB                  1                       5          1.0     1       1
1    

In [7]:
# ── 5D. Mine Sentences_Oare — map column names to akkadian/english ───
# This cell auto-detects which columns contain transliteration and translation

if HAS_SENTENCES:
    cols = [c.lower() for c in sent_df.columns]
    print('Lowercase columns:', cols)

    # Auto-detect akkadian column
    akk_candidates = ['transliteration', 'akkadian', 'translit', 'cuneiform',
                      'sentence', 'text', 'lemma']
    eng_candidates = ['translation', 'english', 'meaning', 'gloss', 'sense']

    akk_col = next((sent_df.columns[i] for i, c in enumerate(cols)
                    if any(k in c for k in akk_candidates)), None)
    eng_col = next((sent_df.columns[i] for i, c in enumerate(cols)
                    if any(k in c for k in eng_candidates)), None)

    print(f'Detected akkadian col: {akk_col}')
    print(f'Detected english col : {eng_col}')

    if akk_col and eng_col:
        sentences_pairs = sent_df[[akk_col, eng_col]].copy()
        sentences_pairs.columns = ['akkadian', 'english']
        sentences_pairs = sentences_pairs.dropna()
        sentences_clean = clean_df(sentences_pairs, 'Sentences_Oare')
        all_frames.append(('sentences_oare', sentences_clean))
        print(f' Added {len(sentences_clean)} pairs from Sentences_Oare')
    else:
        print('  Could not auto-detect columns — printing all unique values of first rows:')
        print(sent_df.head(5).to_string())
        print('\nManually set akk_col and eng_col above and rerun.')

Lowercase columns: ['display_name', 'text_uuid', 'sentence_uuid', 'sentence_obj_in_text', 'translation', 'first_word_transcription', 'first_word_spelling', 'first_word_number', 'first_word_obj_in_text', 'line_number', 'side', 'column']
Detected akkadian col: text_uuid
Detected english col : translation
Sentences_Oare: 9138 clean pairs  (dropped 520 bad, 114 dupes)
✅ Added 9138 pairs from Sentences_Oare


In [8]:
# ── 5E. Mine published_texts.csv ────────────────────────────────────
try:
    pub_df = pd.read_csv(DEEP_PAST_DIR / 'published_texts.csv')
    print('published_texts columns:', pub_df.columns.tolist())
    print('Shape:', pub_df.shape)
    print(pub_df.head(3).to_string())

    cols_pub = [c.lower() for c in pub_df.columns]
    akk_col_p = next((pub_df.columns[i] for i, c in enumerate(cols_pub)
                      if any(k in c for k in ['transliteration','akkadian','translit'])), None)
    eng_col_p = next((pub_df.columns[i] for i, c in enumerate(cols_pub)
                      if any(k in c for k in ['translation','english','meaning'])), None)

    print(f'\nDetected akkadian col: {akk_col_p}')
    print(f'Detected english col : {eng_col_p}')

    if akk_col_p and eng_col_p:
        pub_pairs = pub_df[[akk_col_p, eng_col_p]].copy()
        pub_pairs.columns = ['akkadian', 'english']
        pub_pairs = pub_pairs.dropna()
        pub_clean = clean_df(pub_pairs, 'published_texts')
        all_frames.append(('published_texts', pub_clean))
        print(f' Added {len(pub_clean)} pairs from published_texts')
    else:
        print('  No parallel columns found in published_texts.csv')

except Exception as e:
    print('published_texts.csv error:', e)

published_texts columns: ['oare_id', 'online transcript', 'cdli_id', 'aliases', 'label', 'publication_catalog', 'description', 'genre_label', 'inventory_position', 'online_catalog', 'note', 'interlinear_commentary', 'online_information', 'excavation_no', 'oatp_key', 'eBL_id', 'AICC_translation', 'transliteration_orig', 'transliteration']
Shape: (7953, 19)
                                oare_id                                                      online transcript  cdli_id                  aliases                          label        publication_catalog                                                               description genre_label                     inventory_position online_catalog                                             note                                              interlinear_commentary online_information excavation_no oatp_key eBL_id                          AICC_translation                                                                                            

In [9]:
# ── 5F. Build glossary from eBL Dictionary + OA Lexicon ─────────────
# Used at inference for OOV fallback — NOT added as training pairs
glossary = {}

for fname, label in [('eBL_Dictionary.csv', 'eBL'), ('OA_Lexicon_eBL.csv', 'OA_Lex')]:
    try:
        lex = pd.read_csv(DEEP_PAST_DIR / fname)
        print(f'{label} columns: {lex.columns.tolist()}  shape: {lex.shape}')
        cols_l = [c.lower() for c in lex.columns]
        akk_c = next((lex.columns[i] for i, c in enumerate(cols_l)
                      if any(k in c for k in ['lemma','akkadian','headword','word'])), None)
        eng_c = next((lex.columns[i] for i, c in enumerate(cols_l)
                      if any(k in c for k in ['meaning','english','gloss','sense','translation'])), None)
        if akk_c and eng_c:
            for _, row in lex[[akk_c, eng_c]].dropna().iterrows():
                k = str(row[akk_c]).lower().strip()
                v = str(row[eng_c]).strip()
                if k and v and v.lower() not in ('nan', 'none', ''):
                    glossary[k] = v
            print(f'  → {len(glossary)} glossary entries so far')
        else:
            print(f'  → Could not find akk/eng columns in {fname}')
    except Exception as e:
        print(f'{fname} error: {e}')

print(f'\nTotal glossary entries: {len(glossary)}')

eBL columns: ['word', 'definition', 'derived_from']  shape: (19215, 3)
  → Could not find akk/eng columns in eBL_Dictionary.csv
OA_Lex columns: ['type', 'form', 'norm', 'lexeme', 'eBL', 'I_IV', 'A_D', 'Female(f)', 'Alt_lex']  shape: (39332, 9)
  → Could not find akk/eng columns in OA_Lexicon_eBL.csv

Total glossary entries: 0


In [10]:
# ── 5G. Merge all sources ────────────────────────────────────────────
# Priority: competition > oracc > sentences_oare > published_texts
# Dedup favours higher-priority source

merged_parts = []
source_labels = []
for label, df in all_frames:
    merged_parts.append(df)
    source_labels.extend([label] * len(df))
    print(f'  {label}: {len(df)} pairs')

df_all = pd.concat(merged_parts, ignore_index=True)
df_all['source'] = source_labels

# Dedup on akkadian side (keep first = highest priority)
before = len(df_all)
df_all = df_all.drop_duplicates(subset=['akkadian']).reset_index(drop=True)
print(f'\nTotal after cross-source dedup: {len(df_all)} pairs  (removed {before - len(df_all)} dupes)')
print('\nSource breakdown:')
print(df_all['source'].value_counts().to_string())

  competition: 987 pairs
  oracc: 831 pairs
  sentences_oare: 9138 pairs
  published_texts: 0 pairs

Total after cross-source dedup: 3483 pairs  (removed 7473 dupes)

Source breakdown:
source
sentences_oare    1669
competition        985
oracc              829


In [11]:
# ═══════════════════════════════════════════════════════════════════
# 6. TRAIN / VALIDATION SPLIT
# ═══════════════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split

# Competition pairs get a larger val slice (domain-matched = most valuable eval)
comp_mask  = df_all['source'] == 'competition'
other_mask = ~comp_mask

comp_idx  = df_all[comp_mask].index.tolist()
other_idx = df_all[other_mask].index.tolist()

comp_train_idx,  comp_val_idx  = train_test_split(comp_idx,  test_size=0.12, random_state=SEED)
other_train_idx, other_val_idx = train_test_split(other_idx, test_size=0.05, random_state=SEED)

train_idx = comp_train_idx + other_train_idx
val_idx   = comp_val_idx   + other_val_idx

train_df = df_all.loc[train_idx, ['akkadian', 'english']].reset_index(drop=True)
val_df   = df_all.loc[val_idx,   ['akkadian', 'english']].reset_index(drop=True)
test_raw = pd.read_csv(DEEP_PAST_DIR / 'test.csv').rename(columns={'transliteration': 'akkadian'})

print(f'Train  : {len(train_df)} pairs')
print(f'Val    : {len(val_df)} pairs')
print(f'Test   : {len(test_raw)} pairs (no ground truth)')

Train  : 3239 pairs
Val    : 244 pairs
Test   : 4 pairs (no ground truth)


In [12]:
# ═══════════════════════════════════════════════════════════════════
# 7. DATA AUGMENTATION
# Applied to training set only — triples effective data on small corpora
# ═══════════════════════════════════════════════════════════════════

def augment_swap(df: pd.DataFrame, frac: float = 0.5) -> pd.DataFrame:
    """Adjacent word-swap on Akkadian side — teaches order robustness."""
    rows = []
    for _, row in df.sample(frac=frac, random_state=SEED).iterrows():
        tokens = row['akkadian'].split()
        if len(tokens) > 3:
            i = random.randint(0, len(tokens)-2)
            tokens[i], tokens[i+1] = tokens[i+1], tokens[i]
        rows.append({'akkadian': ' '.join(tokens), 'english': row['english']})
    return pd.DataFrame(rows)

def augment_drop(df: pd.DataFrame, frac: float = 0.3) -> pd.DataFrame:
    """Randomly drop one non-gap token from Akkadian — teaches robustness to gaps."""
    rows = []
    for _, row in df.sample(frac=frac, random_state=SEED+1).iterrows():
        tokens = row['akkadian'].split()
        candidates = [i for i, t in enumerate(tokens) if t not in ('<gap>', 'DEITY', 'PLACE', 'FEM', 'DET')]
        if len(candidates) > 2:
            tokens.pop(random.choice(candidates))
        rows.append({'akkadian': ' '.join(tokens), 'english': row['english']})
    return pd.DataFrame(rows)

aug_swap = augment_swap(train_df, frac=0.5)
aug_drop = augment_drop(train_df, frac=0.3)

train_df_aug = pd.concat([train_df, aug_swap, aug_drop], ignore_index=True)
train_df_aug = train_df_aug.drop_duplicates(subset=['akkadian']).reset_index(drop=True)

print(f'Train before augmentation : {len(train_df)}')
print(f'Train after  augmentation : {len(train_df_aug)}')

Train before augmentation : 3239
Train after  augmentation : 5809


In [13]:
# ═══════════════════════════════════════════════════════════════════
# 8. MODEL SETUP
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'evaluate'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sacrebleu'], check=True)

from transformers import (
    MarianMTModel, MarianTokenizer,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback
)
from datasets import Dataset as HFDataset
import evaluate

MODEL_NAME  = 'Helsinki-NLP/opus-mt-mul-en'
tokenizer   = MarianTokenizer.from_pretrained(MODEL_NAME)
model_mt    = MarianMTModel.from_pretrained(MODEL_NAME)
model_mt.to(device)

print(f'Parameters : {sum(p.numel() for p in model_mt.parameters())//1_000_000}M')
print(f'Vocab size : {tokenizer.vocab_size}')

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Parameters : 143M
Vocab size : 64172


In [14]:
# ═══════════════════════════════════════════════════════════════════
# 9. TOKENISATION
# ═══════════════════════════════════════════════════════════════════
MAX_SRC_LEN = 160
MAX_TGT_LEN = 160

def preprocess_function(examples):
    inputs  = ['>>en<< ' + s for s in examples['akkadian']]
    targets = examples['english']
    model_inputs = tokenizer(inputs,  max_length=MAX_SRC_LEN, truncation=True, padding=False)
    labels       = tokenizer(text_target=targets, max_length=MAX_TGT_LEN, truncation=True, padding=False)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

hf_train = HFDataset.from_pandas(train_df_aug[['akkadian', 'english']])
hf_val   = HFDataset.from_pandas(val_df[['akkadian', 'english']])

tok_train = hf_train.map(preprocess_function, batched=True, remove_columns=['akkadian','english'])
tok_val   = hf_val.map(preprocess_function,   batched=True, remove_columns=['akkadian','english'])

print(f'Tokenised train : {len(tok_train)}')
print(f'Tokenised val   : {len(tok_val)}')

Map:   0%|          | 0/5809 [00:00<?, ? examples/s]

Map:   0%|          | 0/244 [00:00<?, ? examples/s]

Tokenised train : 5809
Tokenised val   : 244


In [20]:
# ═══════════════════════════════════════════════════════════════════
# 10. TRAINING CONFIGURATION
# ═══════════════════════════════════════════════════════════════════

train_dataloader = trainer.get_train_dataloader()
num_training_steps = len(train_dataloader) * int(training_args.num_train_epochs)
num_warmup_steps = int(0.08 * num_training_steps)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = '/kaggle/working/opus_akkadian_v3',
    num_train_epochs            = 40,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_steps                = num_warmup_steps,  
    weight_decay                = 0.01,
    learning_rate               = 5e-5,
    lr_scheduler_type           = 'cosine_with_restarts',
    label_smoothing_factor      = 0.1,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    greater_is_better           = True,
    predict_with_generate       = True,
    generation_max_length       = MAX_TGT_LEN,
    generation_num_beams        = 8,
    fp16                        = use_fp16,
    bf16                        = use_bf16,
    gradient_accumulation_steps = 4,
    max_grad_norm               = 1.0,
    logging_steps               = 25,
    save_total_limit            = 3,
    report_to                   = 'none',
    dataloader_num_workers      = 2,
    seed                        = SEED,
)

trainer = Seq2SeqTrainer(
    model            = model_mt,
    args             = training_args,
    train_dataset    = tok_train,
    eval_dataset     = tok_val,
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=6)],
)

# num_cycles
trainer.create_optimizer_and_scheduler(num_training_steps)
trainer.lr_scheduler = get_scheduler(
    name                = "cosine_with_restarts",
    optimizer           = trainer.optimizer,
    num_warmup_steps    = num_warmup_steps,
    num_training_steps  = num_training_steps,
    scheduler_specific_kwargs = {"num_cycles": 3} 
)

In [21]:
# ── TRAIN ────────────────────────────────────────────────────────────
print('Starting training...')
trainer.train()

trainer.save_model('/kaggle/working/best_opus_akkadian_v3')
tokenizer.save_pretrained('/kaggle/working/best_opus_akkadian_v3')
print(f'\nTraining complete.  Best val BLEU: {trainer.state.best_metric:.4f}')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


Epoch,Training Loss,Validation Loss,Bleu
1,222.178281,7.430586,0.000500
2,51.945693,5.310734,0.006200
3,39.612622,4.433080,0.025900
4,34.280464,3.996875,0.062400
5,31.065540,3.746009,0.087900
6,29.043333,3.592388,0.109200
7,27.330486,3.485451,0.139100
8,26.460801,3.407283,0.164000
9,25.268916,3.347134,0.173800
10,24.366638,3.306899,0.185100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete.  Best val BLEU: 0.2617


In [22]:
# ═══════════════════════════════════════════════════════════════════
# 11. INFERENCE HELPERS
# ═══════════════════════════════════════════════════════════════════
best_tokenizer = MarianTokenizer.from_pretrained('/kaggle/working/best_opus_akkadian_v3')
best_model     = MarianMTModel.from_pretrained('/kaggle/working/best_opus_akkadian_v3').to(device)
best_model.eval()
print('Best model loaded.')

PROPER_NAMES = [
    'Aššur','Assur','Ištar','Istar','Adad','Enlil','Marduk',
    'Šamaš','Shamash','Nanna','Sin','Nergal','Ninurta','Ea',
    'Anu','Babylon','Nineveh','Nippur','Ur','Uruk','Sippar','Lagash','Kish','Kanesh',
]

def remove_output_repetition(text, n=3):
    tokens = text.split()
    if len(tokens) < n*2: return text
    seen, out, i = set(), [], 0
    while i < len(tokens):
        found = False
        for length in range(min(8, len(tokens)-i), n-1, -1):
            gram = tuple(tokens[i:i+length])
            if gram in seen:
                i += length; found = True; break
        if not found:
            for length in range(n, min(9, len(tokens)-i+1)):
                seen.add(tuple(tokens[i:i+length]))
            out.append(tokens[i]); i += 1
    return ' '.join(out)

def postprocess(text):
    text = remove_output_repetition(text.strip(), n=3)
    # Remove repeated sentences
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    seen, result = set(), []
    for p in parts:
        if p.lower().strip() not in seen:
            seen.add(p.lower().strip()); result.append(p)
    text = ' '.join(result)
    # Capitalise sentence starts
    parts2 = re.split(r'(?<=[.!?])\s+', text.strip())
    text = ' '.join(p[0].upper()+p[1:] if p else p for p in parts2)
    # Punctuation spacing
    text = re.sub(r'\s+([.,;:!?])', r'\1', text)
    text = re.sub(r'([.,;:!?])([^\s\'"\)])', r'\1 \2', text)
    text = re.sub(r'\s{2,}', ' ', text).strip()
    # Proper name casing
    for name in PROPER_NAMES:
        text = re.sub(re.escape(name), name, text, flags=re.IGNORECASE)
    return text

def translate_batch(sentences, model, tokenizer,
                    beam_size=8, max_length=MAX_TGT_LEN,
                    repetition_penalty=1.3, no_repeat_ngram=4, length_penalty=1.1):
    prefixed = ['>>en<< ' + normalize_akkadian(s) for s in sentences]
    enc = tokenizer(prefixed, return_tensors='pt', padding=True,
                    truncation=True, max_length=MAX_SRC_LEN).to(device)
    with torch.no_grad():
        generated = model.generate(
            **enc,
            num_beams            = beam_size,
            max_length           = max_length,
            length_penalty       = length_penalty,
            repetition_penalty   = repetition_penalty,
            no_repeat_ngram_size = no_repeat_ngram,
            early_stopping       = True,
        )
    return [postprocess(t) for t in tokenizer.batch_decode(generated, skip_special_tokens=True)]

print('Inference helpers defined.')

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Best model loaded.
Inference helpers defined.


In [23]:
# ═══════════════════════════════════════════════════════════════════
# 12. GENERATE VALIDATION PREDICTIONS
# ═══════════════════════════════════════════════════════════════════
print('Generating validation predictions...')
BATCH_SIZE = 64
val_preds  = []

for start in range(0, len(val_df), BATCH_SIZE):
    batch = val_df['akkadian'].iloc[start:start+BATCH_SIZE].tolist()
    val_preds.extend(translate_batch(batch, best_model, best_tokenizer))
    if (start // BATCH_SIZE + 1) % 5 == 0:
        print(f'  {min(start+BATCH_SIZE, len(val_df))}/{len(val_df)}')

val_refs = val_df['english'].tolist()
print(f'Done. {len(val_preds)} predictions.')

Generating validation predictions...
Done. 244 predictions.


In [24]:
# ═══════════════════════════════════════════════════════════════════
# 13. ALL EVALUATION METRICS
# ═══════════════════════════════════════════════════════════════════
import sacrebleu as scb

# ── BLEU ──
bleu_result  = scb.corpus_bleu(val_preds, [val_refs], tokenize='13a')
bleu_score   = bleu_result.score / 100

# ── METEOR ──
meteor_metric = evaluate.load('meteor')
meteor_score  = meteor_metric.compute(predictions=val_preds, references=val_refs)['meteor']

# ── ROUGE ──
rouge_metric = evaluate.load('rouge')
rouge_result = rouge_metric.compute(predictions=val_preds, references=val_refs)
rougeL_score = rouge_result['rougeL']

# ── TER ──
ter_result = scb.corpus_ter(val_preds, [val_refs])
ter_score  = ter_result.score / 100

# ── chrF / chrF++ ──
chrf_score   = scb.corpus_chrf(val_preds, [val_refs], word_order=0).score / 100
chrfpp_score = scb.corpus_chrf(val_preds, [val_refs], word_order=2).score / 100

# ── BERTScore ──
from bert_score import score as bert_score_fn
print('Computing BERTScore...')
P, R, F1 = bert_score_fn(
    cands=val_preds, refs=val_refs,
    model_type='distilbert-base-uncased',
    lang='en', verbose=False, rescale_with_baseline=True,
)
bertscore_F1 = F1.mean().item()

print('Metrics computed.')

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Computing BERTScore...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Metrics computed.


In [25]:
# ═══════════════════════════════════════════════════════════════════
# 14. RESULTS TABLE
# ═══════════════════════════════════════════════════════════════════
print('\n' + '═'*68)
print('  TABLE 1: Akkadian→English MT — Validation Results')
print(f'  Model : Helsinki-NLP/opus-mt-mul-en (fine-tuned v3)')
print(f'  Data  : Deep Past + ORACC + Sentences_Oare + published_texts')
print('─'*68)
print(f'  {"Metric":<20} {"Score":>10}  Notes')
print('─'*68)
print(f'  {"BLEU":<20} {bleu_score:>10.4f}  sacrebleu 13a | target ≥0.32')
print(f'  {"METEOR":<20} {meteor_score:>10.4f}  recall-weighted, synonym-aware')
print(f'  {"ROUGE-L":<20} {rougeL_score:>10.4f}  LCS F1')
print(f'  {"ROUGE-1":<20} {rouge_result["rouge1"]:>10.4f}')
print(f'  {"ROUGE-2":<20} {rouge_result["rouge2"]:>10.4f}')
print(f'  {"TER":<20} {ter_score:>10.4f}  lower=better')
print(f'  {"chrF":<20} {chrf_score:>10.4f}  char n-gram F-score')
print(f'  {"chrF++":<20} {chrfpp_score:>10.4f}  + word bigrams')
print(f'  {"BERTScore F1":<20} {bertscore_F1:>10.4f}  rescaled distilbert-base')
print('─'*68)
print(f'  Train pairs : {len(train_df_aug)}  |  Val pairs : {len(val_df)}')
print(f'  BLEU target : 0.32  →  {"✅ MET" if bleu_score >= 0.32 else "❌ NOT MET — see notes below"}')
print('═'*68)

results = {
    'model': MODEL_NAME, 'version': 'v3',
    'train_pairs': len(train_df_aug), 'val_pairs': len(val_df),
    'bleu': round(bleu_score,4), 'meteor': round(meteor_score,4),
    'rougeL': round(rougeL_score,4), 'ter': round(ter_score,4),
    'chrf': round(chrf_score,4), 'chrfpp': round(chrfpp_score,4),
    'bertscore_f1': round(bertscore_F1,4),
}
with open('/kaggle/working/eval_results_v3.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved: /kaggle/working/eval_results_v3.json')


════════════════════════════════════════════════════════════════════
  TABLE 1: Akkadian→English MT — Validation Results
  Model : Helsinki-NLP/opus-mt-mul-en (fine-tuned v3)
  Data  : Deep Past + ORACC + Sentences_Oare + published_texts
────────────────────────────────────────────────────────────────────
  Metric                    Score  Notes
────────────────────────────────────────────────────────────────────
  BLEU                     0.1823  sacrebleu 13a | target ≥0.32
  METEOR                   0.3247  recall-weighted, synonym-aware
  ROUGE-L                  0.3837  LCS F1
  ROUGE-1                  0.4357
  ROUGE-2                  0.2804
  TER                      0.7849  lower=better
  chrF                     0.3880  char n-gram F-score
  chrF++                   0.3724  + word bigrams
  BERTScore F1             0.5411  rescaled distilbert-base
────────────────────────────────────────────────────────────────────
  Train pairs : 5809  |  Val pairs : 244
  BLEU target : 0.3

In [26]:
# ═══════════════════════════════════════════════════════════════════
# 15. PER-SENTENCE ANALYSIS
# ═══════════════════════════════════════════════════════════════════
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smooth = SmoothingFunction().method1

per_sent = []
for ref, pred in zip(val_refs, val_preds):
    rt = ref.lower().split(); pt = pred.lower().split()
    per_sent.append(sentence_bleu([rt], pt, smoothing_function=smooth) if rt and pt else 0.0)

val_analysis = val_df.copy()
val_analysis['prediction'] = val_preds
val_analysis['sent_bleu']  = per_sent

print('=== TOP 5 TRANSLATIONS ===')
for _, row in val_analysis.nlargest(5, 'sent_bleu').iterrows():
    print(f'  SRC : {row["akkadian"][:90]}')
    print(f'  REF : {row["english"][:90]}')
    print(f'  PRED: {row["prediction"][:90]}')
    print(f'  BLEU: {row["sent_bleu"]:.3f}  ' + '─'*50)

print('\n=== WORST 5 TRANSLATIONS ===')
for _, row in val_analysis.nsmallest(5, 'sent_bleu').iterrows():
    print(f'  SRC : {row["akkadian"][:90]}')
    print(f'  REF : {row["english"][:90]}')
    print(f'  PRED: {row["prediction"][:90]}')
    print(f'  BLEU: {row["sent_bleu"]:.3f}  ' + '─'*50)

=== TOP 5 TRANSLATIONS ===
  SRC : a na a lá hi im KISIB a mur sa ra sà
  REF : To Ali-ahum; seal of Amur-šarrassa.
  PRED: To Ali-ahum; sEal of AmUr-šarrassa.
  BLEU: 1.000  ──────────────────────────────────────────────────
  SRC : 10 ma na KÙ.BABBAR sa ru pá am a mur ISTAR DUMU da mu ma i na sí im SÍG.HI.A sa ú sí na la
  REF : Amur-Ištar son of Damuma has received 10 minas of refined silver from the proceeds of Ušin
  PRED: AmUr-Ištar son of Damuma has received 10 minas of refined silver from the proceeds of Ušin
  BLEU: 1.000  ──────────────────────────────────────────────────
  SRC : ta ah sí sà tum sa sí be sa sí im SÍG.HI.A sa ú sí na lam
  REF : Memoranda with witnesses concerning the proceeds of Ušinalam's wool.
  PRED: Memoranda with witnesses concerning the proceeds of Ušinalam's wool.
  BLEU: 1.000  ──────────────────────────────────────────────────
  SRC : 3 GÍN KÙ.BABBAR a na e na ah DINGIR 0.25 GÍN a na sa ra qí tim a na AH.ME ri im 0.1666 GÍN
  REF : 3 shekels of silve

In [29]:
# ═══════════════════════════════════════════════════════════════════
# 16. FINAL SUBMISSION — 4 competition test rows
# ═══════════════════════════════════════════════════════════════════
test_df = test_raw.copy()
test_df['akkadian_clean'] = test_df['akkadian'].apply(normalize_akkadian)

test_preds = translate_batch(
    test_df['akkadian_clean'].tolist(),
    best_model, best_tokenizer,
    beam_size          = 10,
    repetition_penalty = 1.4,
    no_repeat_ngram    = 4,
    length_penalty     = 1.2,
)

print('\n=== COMPETITION TEST PREDICTIONS ===')
for i, (src, pred) in enumerate(zip(test_df['akkadian'].tolist(), test_preds)):
    print(f'\n[{i}] SRC : {src[:130]}')
    print(f'    PRED: {pred}')

id_col = 'id' if 'id' in test_df.columns else test_df.columns[0]
submission = pd.DataFrame({'id': test_df[id_col].values, 'translation': test_preds})
submission.to_csv('/kaggle/working/submission_v3.csv', index=False, encoding='utf-8-sig')
print(f'\n Submission saved: /kaggle/working/submission.csv')
print(submission)


=== COMPETITION TEST PREDICTIONS ===

[0] SRC : um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi„-ma mup-pu-um aa a-lim(ki) i-li-
    PRED: From the Kanesh colony to the City: Here the karkarru and the station is on the way to Pūšu-kēn.

[1] SRC : i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-nim ma-ma-an KÙ.AN i-aa-ú-mu-ni i-na né-mì-lim da-aùr ú-lá e-WA ia-ra-tí-au kà-ru-um 
    PRED: In my letter I shall mention to you (plUral). As to the mina of silver that I owed to you, in the name of Iddin-AššUr, no one will take it away from you. The Kanesh colony took away my silver.

[2] SRC : ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É.GAL-lim i-dí-in lu té-ra-at É.GAL-lim ú-kà-lim lu na-aí-ma a-dí-ni 
    PRED: From oUr letter we have here once or twice to the colony. It is up to you to comply with the wishes of the colony. Do not let the silver stop at my expense. My father and my father have been embroiled in a lawsuit an

In [30]:
# ═══════════════════════════════════════════════════════════════════
# 17. REPETITION AUDIT
# ═══════════════════════════════════════════════════════════════════
def repetition_rate(text, n=4):
    toks = text.lower().split()
    if len(toks) < n: return 0.0
    ngrams = [tuple(toks[i:i+n]) for i in range(len(toks)-n+1)]
    repeated = sum(v-1 for v in Counter(ngrams).values() if v > 1)
    return repeated / max(len(ngrams), 1)

val_rep  = [repetition_rate(p) for p in val_preds]
test_rep = [repetition_rate(p) for p in test_preds]

print(f'Validation repetition (mean): {np.mean(val_rep):.4f}')
print(f'Validation repetition (max) : {np.max(val_rep):.4f}')
print(f'Val rows with rep > 0.1     : {sum(r>0.1 for r in val_rep)}')
print()
for i, (pred, rep) in enumerate(zip(test_preds, test_rep)):
    print(f'Test [{i}] rep={rep:.3f}  {pred[:80]}')

Validation repetition (mean): 0.0000
Validation repetition (max) : 0.0000
Val rows with rep > 0.1     : 0

Test [0] rep=0.000  From the Kanesh colony to the City: Here the karkarru and the station is on the 
Test [1] rep=0.000  In my letter I shall mention to you (plUral). As to the mina of silver that I ow
Test [2] rep=0.000  From oUr letter we have here once or twice to the colony. It is up to you to com
Test [3] rep=0.000  We entered Karkarša's house to settle the case, and the man has nothing to do wi


## Research Notes

### Why BLEU was 0.10 in v2
- Only **1,654 training pairs** — far too few for a sequence-to-sequence model to generalise
- The `Sentences_Oare_FirstWord_LinNum.csv` and other files were unused
- No data augmentation

### What v3 does differently
| Change | Expected BLEU gain |
|--------|-------------------|
| All datasets mined (target 5k–15k pairs) | +0.08–0.15 |
| Data augmentation (swap + drop, +80% pairs) | +0.03–0.06 |
| 40 epochs + cosine restarts | +0.03–0.05 |
| Smaller batch (more gradient steps) | +0.02–0.04 |
| Label smoothing ε=0.1 | +0.01–0.03 |

### Metric interpretation for Akkadian
- **chrF++** is the most reliable metric for Akkadian — character-level, handles morphological variants
- **BLEU** penalises valid paraphrases harshly — treat as a lower bound
- **BERTScore** captures semantic similarity that BLEU misses
- **TER < 0.7** is the practical goal; below 0.5 indicates good fluency

### If BLEU is still below 0.32 after this run
The remaining lever is switching backbone to `facebook/mbart-large-50-many-to-one-mmt` (600M params, stronger multilingual pretraining). It requires ~4× more GPU memory but consistently outperforms MarianMT on low-resource pairs with < 5k training examples.